# Crop Disease Detection using MobileNetV2

This notebook trains a Convolutional Neural Network (CNN) based on MobileNetV2 using the PlantVillage dataset. The goal is to classify 38 different plant diseases across 14 crops with 95%+ accuracy.

In [ ]:
!pip install -q tensorflow-datasets

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow version:", tf.__version__)

## 1. Load the PlantVillage Dataset

In [ ]:
# Load the dataset using TFDS
# PlantVillage dataset has 38 classes.
dataset, info = tfds.load('plant_village', with_info=True, as_supervised=True)

train_ds, val_ds = dataset['train'], dataset['train'] # Plant village only has 'train' split by default

# We will split the dataset into 80% train and 20% validation manually
DATASET_SIZE = info.splits['train'].num_examples
train_size = int(0.8 * DATASET_SIZE)
val_size = int(0.2 * DATASET_SIZE)

full_ds = dataset['train'].shuffle(10000, seed=42)
train_ds = full_ds.take(train_size)
val_ds = full_ds.skip(train_size)

class_names = info.features['label'].names
NUM_CLASSES = len(class_names)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Training samples: {train_size}")
print(f"Validation samples: {val_size}")

## 2. Preprocess Data

In [ ]:
IMG_SIZE = 224 # MobileNetV2 expects 224x224
BATCH_SIZE = 32

def format_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0 # Normalize to [0,1]
    return image, label

train_batches = train_ds.shuffle(1000).map(format_image).batch(BATCH_SIZE).prefetch(1)
validation_batches = val_ds.map(format_image).batch(BATCH_SIZE).prefetch(1)

## 3. Build the Model (Transfer Learning with MobileNetV2)

In [ ]:
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

# Create the base model from the pre-trained model MobileNet V2
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                               include_top=False,
                                               weights='imagenet')

# Freeze the base_model
base_model.trainable = False

# Add classification head
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
prediction_layer = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')

model = tf.keras.Sequential([
  base_model,
  global_average_layer,
  tf.keras.layers.Dropout(0.2),
  prediction_layer
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 4. Train the Model

In [ ]:
EPOCHS = 10

# Use EarlyStopping and ModelCheckpoint to save the best model
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('plant_disease_model.h5', save_best_only=True)
]

history = model.fit(train_batches,
                    epochs=EPOCHS,
                    validation_data=validation_batches,
                    callbacks=callbacks)

## 5. Evaluate and Save

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

In [ ]:
# Save class names for inference later
import json
with open('class_indices.json', 'w') as f:
    json.dump(class_names, f)

print("Model saved as plant_disease_model.h5")
print("Class indices saved as class_indices.json")

## 6. Download Files to Local Machine
Run this cell to download the trained model and class indices if you are on Google Colab.

In [ ]:
try:
  from google.colab import files
  files.download('plant_disease_model.h5')
  files.download('class_indices.json')
except:
  print("Not running in Colab, skip downloading.")